<a href="https://colab.research.google.com/github/rafaelrubo/extensao-cs-qualidade-do-ar/blob/main/extens%C3%A3o_qualidade_do_ar-02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================
# INSTALAÇÃO DAS BIBLIOTECAS
# =========================================
!pip install pandas folium openpyxl -q

In [ ]:
# =========================================
# IMPORTAÇÃO DAS BIBLIOTECAS
# =========================================
import pandas as pd
import folium
import re
from google.colab import files

In [ ]:
# =========================================
# FUNÇÃO DMS -> DECIMAL
# =========================================
def dms_to_decimal(dms):
    pattern = r"(\d+)°(\d+)'([\d\.]+)\"([NSEW])"
    match = re.match(pattern, dms)

    if not match:
        return None

    degrees, minutes, seconds, direction = match.groups()

    decimal = (
        float(degrees)
        + float(minutes) / 60
        + float(seconds) / 3600
    )

    if direction in ['S', 'W']:
        decimal *= -1

    return decimal

In [ ]:
# =========================================
# MAPA DE CORES
# =========================================
cores = {
    'verde': 'green',
    'amarelo': 'orange',
    'laranja': 'darkorange',
    'vermelho': 'red'
}

In [ ]:
# =========================================
# UPLOAD DO ARQUIVO
# =========================================
uploaded = files.upload()

arquivo = list(uploaded.keys())[0]

Saving 20270427 - medidas.xlsx to 20270427 - medidas (2).xlsx


In [ ]:
# =========================================
# LEITURA DA PLANILHA
# =========================================
df = pd.read_excel(
    arquivo,
    header=1
)

In [ ]:
# =========================================
# FILTRAR AMBIENTES
# =========================================
ambientes_excluir = [
    'porta de entrada',
    'praça de alimentação'
]

# normalização

df['Ambiente_norm'] = (
    df['Ambiente']
    .astype(str)
    .str.strip()
    .str.lower()
)


df = df[
    ~df['Ambiente_norm'].isin(ambientes_excluir)
].copy()

In [ ]:
# =========================================
# CONVERTER COORDENADAS
# =========================================
df['lat'] = df['Latitude'].apply(dms_to_decimal)
df['lon'] = df['Longitude'].apply(dms_to_decimal)

In [ ]:
# =========================================
# ORDENAR PELO PONTO
# =========================================
df = df.sort_values('Ponto')

In [ ]:
# =========================================
# CRIAR MAPA
# =========================================
mapa = folium.Map(
    location=[df['lat'].mean(), df['lon'].mean()],
    zoom_start=10,
    tiles='CartoDB positron'
)

In [ ]:
# =========================================
# ADICIONAR CÍRCULOS COLORIDOS
# =========================================
coords_linha = []

for _, row in df.iterrows():

    cor = cores.get(
        str(row['Qualidade']).lower(),
        'gray'
    )

    coords_linha.append([row['lat'], row['lon']])

    popup = f"""
    <b>Ponto:</b> {row['Ponto']}<br>
    <b>Nome:</b> {row['Nome']}<br>
    <b>Ambiente:</b> {row['Ambiente']}<br>
    <b>Qualidade:</b> {row['Qualidade']}<br>
    <b>CO₂:</b> {row['CO₂ (ppm)']} ppm
    """

    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=8,
        color=cor,
        fill=True,
        fill_color=cor,
        fill_opacity=0.8,
        popup=popup,
        tooltip=f"Ponto {row['Ponto']}"
    ).add_to(mapa)

In [ ]:
# =========================================
# LINHA DE EVOLUÇÃO
# =========================================
folium.PolyLine(
    coords_linha,
    color='blue',
    weight=2,
    opacity=0.7
).add_to(mapa)

In [ ]:
# =========================================
# SALVAR MAPA
# =========================================
saida = 'mapa_evolucao_medidas.html'
mapa.save(saida)

print(f'Mapa salvo: {saida}')

Mapa salvo: mapa_evolucao_medidas.html
